# 📊 Amazon Sales Analytics - Complete Analysis
**Dataset**: 100,000 Amazon transactions (2020-2024)  
**Analysis**: EDA, Statistics, SQL, and Business Insights

---

## 🎯 Executive Summary

| Metric | Value |
|--------|-------|
| **Total Revenue** | $91.8M |
| **Total Orders** | 100,000 |
| **Average Order Value** | $918 |
| **Unique Customers** | 43,233 |
| **Date Range** | 2020-01 to 2024-12 |

### Key Findings
1. 💰 **Revenue Concentration**: Top 20% customers generate 44% of revenue
2. 📦 **Category Performance**: Electronics & Sports lead (~17K orders each)
3. ⚠️ **Order Status**: 25% orders not delivered (cancelled/returned/pending)
4. 💳 **Payment**: Credit cards dominate (35%), Cash on Delivery only 5%
5. 🌍 **Geography**: 70% sales from US, Texas & California are top states

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"✅ Analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

---
## 1️⃣ Data Loading & Overview

In [ ]:
# Load data
df = pd.read_csv('../data/Amazon.csv')
df.columns = df.columns.str.lower()
df['orderdate'] = pd.to_datetime(df['orderdate'])

# Create SQLite connection for SQL analysis
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')

print(f"📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n📋 Columns: {', '.join(df.columns)}")
print(f"\n✅ Missing Values: {df.isnull().sum().sum()} (None!)")
print(f"📅 Date Range: {df['orderdate'].min().date()} to {df['orderdate'].max().date()}")

In [ ]:
# Quick Profile
print("\n" + "="*60)
print("NUMERICAL COLUMNS")
print("="*60)
numerical = ['quantity', 'unitprice', 'discount', 'tax', 'shippingcost', 'totalamount']
print(df[numerical].describe().round(2))

print("\n" + "="*60)
print("CATEGORICAL COLUMNS")
print("="*60)
categorical = ['category', 'paymentmethod', 'orderstatus', 'country']
for col in categorical:
    print(f"\n{col.upper()}:")
    print(df[col].value_counts().head(5))

---
## 2️⃣ SQL Analysis

In [ ]:
# SQL: Revenue by Category
query1 = '''
SELECT 
    category,
    COUNT(*) as orders,
    SUM(quantity) as items,
    ROUND(SUM(totalamount), 2) as revenue,
    ROUND(AVG(totalamount), 2) as aov
FROM sales 
GROUP BY category
ORDER BY revenue DESC
'''

revenue_by_cat = pd.read_sql(query1, conn)
print("💰 REVENUE BY CATEGORY:")
print(revenue_by_cat.to_string(index=False))

In [ ]:
# SQL: Top Customers
query2 = '''
SELECT 
    customerid,
    COUNT(*) as orders,
    ROUND(SUM(totalamount), 2) as lifetime_value,
    ROUND(AVG(totalamount), 2) as avg_order
FROM sales 
GROUP BY customerid
ORDER BY lifetime_value DESC
LIMIT 10
'''

top_customers = pd.read_sql(query2, conn)
print("👑 TOP 10 CUSTOMERS BY LIFETIME VALUE:")
print(top_customers.to_string(index=False))

In [ ]:
# SQL: Monthly Trend
query3 = '''
SELECT 
    SUBSTR(orderdate, 1, 7) as month,
    COUNT(*) as orders,
    ROUND(SUM(totalamount), 2) as revenue
FROM sales 
GROUP BY month
ORDER BY month
'''

monthly = pd.read_sql(query3, conn)
monthly['month'] = pd.to_datetime(monthly['month'])

# Plot monthly trend
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.plot(monthly['month'], monthly['revenue'], marker='o', linewidth=2, markersize=4)
ax1.set_title('📈 Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax1.set_ylabel('Revenue ($)')
ax1.grid(True, alpha=0.3)

ax2.bar(monthly['month'], monthly['orders'], color='steelblue', alpha=0.7)
ax2.set_title('📦 Monthly Order Volume', fontsize=14, fontweight='bold')
ax2.set_ylabel('Orders')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📊 Average Monthly Revenue: ${monthly['revenue'].mean():,.0f}")
print(f"📊 Revenue Std Dev: ${monthly['revenue'].std():,.0f}")

---
## 3️⃣ Visual Analysis

In [ ]:
# Category Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Revenue by Category
cat_revenue = df.groupby('category')['totalamount'].sum().sort_values(ascending=True)
cat_revenue.plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('💰 Revenue by Category', fontweight='bold')
axes[0,0].set_xlabel('Revenue ($)')

# 2. Order Status Distribution
status_counts = df['orderstatus'].value_counts()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']
axes[0,1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%', 
              colors=colors, startangle=90)
axes[0,1].set_title('📋 Order Status Distribution', fontweight='bold')

# 3. Payment Methods
payment_counts = df['paymentmethod'].value_counts()
payment_counts.plot(kind='bar', ax=axes[1,0], color='coral')
axes[1,0].set_title('💳 Payment Methods', fontweight='bold')
axes[1,0].set_ylabel('Count')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Top States
state_revenue = df.groupby('state')['totalamount'].sum().sort_values(ascending=False).head(10)
state_revenue.plot(kind='bar', ax=axes[1,1], color='mediumseagreen')
axes[1,1].set_title('🗺️ Top 10 States by Revenue', fontweight='bold')
axes[1,1].set_ylabel('Revenue ($)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Customer Analysis
customer_stats = df.groupby('customerid').agg({
    'totalamount': ['sum', 'mean', 'count']
}).round(2)
customer_stats.columns = ['lifetime_value', 'avg_order', 'order_count']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Order Value Distribution
axes[0].hist(df['totalamount'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df['totalamount'].mean(), color='red', linestyle='--', 
                label=f"Mean: ${df['totalamount'].mean():.0f}")
axes[0].axvline(df['totalamount'].median(), color='green', linestyle='--', 
                label=f"Median: ${df['totalamount'].median():.0f}")
axes[0].set_title('💵 Order Value Distribution', fontweight='bold')
axes[0].set_xlabel('Order Value ($)')
axes[0].legend()

# Customer Order Frequency
freq_counts = customer_stats['order_count'].value_counts().sort_index()
freq_counts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('🔄 Customer Order Frequency', fontweight='bold')
axes[1].set_xlabel('Number of Orders')
axes[1].set_ylabel('Number of Customers')

# Customer Lifetime Value Distribution
axes[2].hist(customer_stats['lifetime_value'], bins=50, color='lightgreen', 
             edgecolor='black', alpha=0.7)
axes[2].set_title('👤 Customer Lifetime Value', fontweight='bold')
axes[2].set_xlabel('Lifetime Value ($)')

plt.tight_layout()
plt.show()

print(f"👥 Total Unique Customers: {len(customer_stats):,}")
print(f"📊 Avg Orders per Customer: {customer_stats['order_count'].mean():.2f}")
print(f"💰 Avg Customer LTV: ${customer_stats['lifetime_value'].mean():,.2f}")

---
## 4️⃣ Statistical Insights

In [ ]:
from scipy import stats

print("="*60)
print("📊 STATISTICAL ANALYSIS")
print("="*60)

order_values = df['totalamount']

# Central Tendency
print(f"\n📌 CENTRAL TENDENCY:")
print(f"   Mean: ${order_values.mean():,.2f}")
print(f"   Median: ${order_values.median():,.2f}")
print(f"   Mode: ${order_values.mode()[0]:,.2f}")

# Dispersion
print(f"\n📌 DISPERSION:")
print(f"   Std Dev: ${order_values.std():,.2f}")
print(f"   Coefficient of Variation: {(order_values.std()/order_values.mean())*100:.1f}%")

# Percentiles
percentiles = [25, 50, 75, 90, 95, 99]
print(f"\n📌 PERCENTILES:")
for p in percentiles:
    print(f"   {p}th: ${np.percentile(order_values, p):,.2f}")

# Skewness & Kurtosis
skewness = stats.skew(order_values)
print(f"\n📌 DISTRIBUTION SHAPE:")
print(f"   Skewness: {skewness:.3f} ({'Right-skewed' if skewness > 0 else 'Left-skewed'})")
print(f"   Kurtosis: {stats.kurtosis(order_values):.3f}")

In [ ]:
# Pareto Analysis (80/20 Rule)
customer_revenue = df.groupby('customerid')['totalamount'].sum().sort_values(ascending=False)
cumulative_pct = customer_revenue.cumsum() / customer_revenue.sum() * 100
customer_pct = np.arange(1, len(customer_revenue) + 1) / len(customer_revenue) * 100

# Find the 80% revenue point
idx_80 = np.where(cumulative_pct >= 80)[0][0]
customers_for_80 = customer_pct[idx_80]

print("="*60)
print("📊 PARETO ANALYSIS (80/20 RULE)")
print("="*60)
print(f"\n👑 Top 20% of customers generate: {cumulative_pct[int(len(customer_revenue)*0.2)]:.1f}% of revenue")
print(f"📈 80% of revenue comes from: {customers_for_80:.1f}% of customers")

# Visualize Pareto
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(customer_pct[:1000], customer_revenue.values[:1000], color='steelblue', alpha=0.7)
ax1.set_xlabel('Customer Percentile (%)', fontsize=12)
ax1.set_ylabel('Revenue per Customer ($)', fontsize=12, color='steelblue')

ax2 = ax1.twinx()
ax2.plot(customer_pct, cumulative_pct, color='red', linewidth=2, label='Cumulative %')
ax2.axhline(y=80, color='green', linestyle='--', alpha=0.7, label='80% Revenue')
ax2.axvline(x=customers_for_80, color='green', linestyle='--', alpha=0.7)
ax2.set_ylabel('Cumulative Revenue (%)', fontsize=12, color='red')
ax2.legend(loc='center right')

plt.title('📊 Pareto Analysis: Customer Revenue Concentration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5️⃣ Business Recommendations

### 🎯 Strategic Recommendations

Based on the analysis, here are data-driven recommendations:

| Priority | Recommendation | Expected Impact |
|----------|---------------|-----------------|
| **HIGH** | 🎯 **VIP Customer Program** - Focus on top 20% customers who drive 44% revenue | Retain high-value customers, reduce churn risk |
| **HIGH** | ⚠️ **Reduce Order Issues** - 25% orders not delivered (cancelled/returned/pending) | Improve customer satisfaction, recover ~$23M revenue |
| **MEDIUM** | 📦 **Inventory Focus** - Prioritize Electronics & Sports categories | Maximize revenue from top-performing categories |
| **MEDIUM** | 🌍 **Geographic Expansion** - Texas & California are 45% of sales; expand in underperforming states | Diversify revenue geographically |
| **LOW** | 💳 **Payment Optimization** - Cash on Delivery is only 5%; consider incentives for digital payments | Reduce cash handling costs |

In [ ]:
# Calculate potential revenue recovery
non_delivered = df[df['orderstatus'].isin(['Cancelled', 'Returned', 'Pending'])]
potential_recovery = non_delivered['totalamount'].sum()

print("="*60)
print("💡 REVENUE OPPORTUNITY ANALYSIS")
print("="*60)
print(f"\n⚠️  Non-Delivered Orders:")
print(f"   Count: {len(non_delivered):,} ({len(non_delivered)/len(df)*100:.1f}%)")
print(f"   Value: ${potential_recovery:,.2f}")
print(f"\n💰 If 50% issues resolved: +${potential_recovery*0.5:,.2f} revenue")

# Category breakdown of issues
print(f"\n📊 Non-Delivered Orders by Category:")
print(non_delivered['category'].value_counts())

---
## ✅ Analysis Complete

**Summary**: This analysis examined 100,000 Amazon transactions to identify revenue patterns, customer behavior, and operational opportunities. The data reveals a mature business with concentrated revenue among top customers and significant opportunity to improve order fulfillment.

**Tools Used**: Python (Pandas, Matplotlib, Seaborn), SQLite, Statistics (SciPy)